In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import json
import pickle

In [2]:
# Load Yijun's 5-fold indices
kfold_pt = '/home/yaoyi/shared/aksdb-covars/aksdb-point-data/experimental_data/sfold5_split_train_test_indices.json'
with open(kfold_pt, 'r') as file:
    kfold_indices = json.load(file)
print(type(kfold_indices))

<class 'dict'>


KFOLD INDICES STRUCTURE:
Dict
- first level are keys labeled fold_0 to fold_1
    - in each fold_i value is another dictionary where the keys are 'train' and 'test'
        - the value is a list of indices

In [3]:
# Load in the geojson with data
pt_data_pt = '../data/point_pixel_satbands_gt_epsg4326_v3_removednodata.geojson'
point_data_gdf = gpd.read_file(pt_data_pt)
print(point_data_gdf.columns)

ERROR 1: PROJ: proj_create_from_database: Open of /users/0/chen7924/.conda/envs/gp2/share/proj failed


Index(['id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'x_pixel', 'y_pixel', 'band_1', 'band_2',
       'band_3', 'band_4', 'band_5', 'band_6', 'band_7', 'band_8', 'band_9',
       'band_10', 'band_11', 'band_12', 'band_13', 'band_14', 'band_15',
       'band_16', 'band_17', 'band_18', 'band_19', 'band_20', 'band_21',
       'band_22', 'band_23', 'band_24', 'band_25', 'band_26', 'band_27',
       'band_28', 'band_29', 'band_30', 'band_31', 'band_32', 'band_33',
       'band_34', 'band_35', 'band_36', 'band_37', 'band_38', 'band_39',
       'band_40', 'band_41', 'band_42', 'band_43', 'band_44', 'band_45',
       'band_46', 'band_47', 'band_48', 'band_49', 'band_50', 'band_51',
       'band_52', 'band_53', 'band_54', 'band_55', 'band_56', 'band_57',
       'band_58', 'band_59', 'band_60', 'aksdb_pf1m_bin', 'aksdb_pf1m_bin_gt',
       'geometry'],
      dtype='object')


In [4]:
# Define random forest run function
def run_rf(all_ids, kfold_indices, X, y, hyperparameters, save_preds=False, save_rf=False):
    indices = np.array(all_ids)
    
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_precision_0 = []
    fold_precision_1 = []
    fold_recall_0 = []
    fold_recall_1 = []
    
    for fold, train_test_dict in kfold_indices.items():
        print('Training Fold: ', fold)

        train_indices = train_test_dict['train']
        test_indices = train_test_dict['test']

        train_rows = np.where(np.isin(indices, train_indices))[0]
        test_rows = np.where(np.isin(indices, test_indices))[0]

        X_train = X[train_rows]
        X_test = X[test_rows]
        y_train = y[train_rows]
        y_test = y[test_rows] 

        print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
        print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

        est = hyperparameters['est']
        verbose = hyperparameters['verbose']
        n_jobs = hyperparameters['n_jobs']
        rf = RandomForestClassifier(n_estimators=est, verbose=verbose, n_jobs=n_jobs)
        rf.fit(X_train, y_train.ravel())
        y_pred = rf.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='binary')
        recall = recall_score(y_test, y_pred, average='binary')
        class_precisions = precision_score(y_test, y_pred, average=None)
        class_recalls = recall_score(y_test, y_pred, average=None)
        
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_precision_0.append(class_precisions[0])
        fold_precision_1.append(class_precisions[1])
        fold_recall_0.append(class_recalls[0])
        fold_recall_1.append(class_recalls[1])
        
        fold_num = int(fold.split('_')[-1])
        print(f"Fold {fold_num} Accuracy: {accuracy:.4f}")
        print(f"Fold {fold_num} Precision: {precision:.4f}")
        print(f"Fold {fold_num} Recall: {recall:.4f}")
        
        if save_preds:
            test_ids = indices[np.isin(indices, test_indices)]
            print(len(test_ids))
            print(len(test_indices))
            preds = {
                "id": test_ids,  
                "gt": y_test.flatten(), 
                "pred": y_pred  
            }
            df_predictions = pd.DataFrame(preds)
            preds_outpath = f"test_predictions_{fold}.csv"
            df_predictions.to_csv(preds_outpath, index=False)

            print(f"Test predictions saved to {preds_outpath}")
            
        if save_rf:
            model_file = f"rf_weights_{fold}.pkl"
            with open(model_file, "wb") as f:
                pickle.dump(rf, f)
            print(f"Model saved to {model_file}")
        
    return (
        fold_accuracies, fold_precisions, fold_recalls,
        fold_precision_0, fold_precision_1,
        fold_recall_0, fold_recall_1
    )

## Experiment Random Forest 1 (RF-1)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B
    - Covariates: Lat, Lon
    - Order: B, G, R, Lon, Lat
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None


In [5]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (37249, 72)
After removing NaN:  (14997, 72)


In [6]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

X shape:  (14997, 5)
Y shape:  (14997, 1)


In [7]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [8]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-1_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (12060, 5), X_test shape: (2937, 5)
y_train shape: (12060, 1), y_test shape: (2937, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.9094
Fold 0 Precision: 0.8447
Fold 0 Recall: 0.7603
Training Fold:  fold_1
X_train shape: (11777, 5), X_test shape: (3220, 5)
y_train shape: (11777, 1), y_test shape: (3220, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 1 Accuracy: 0.9202
Fold 1 Precision: 0.8003
Fold 1 Recall: 0.7966
Training Fold:  fold_2
X_train shape: (12271, 5), X_test shape: (2726, 5)
y_train shape: (12271, 1), y_test shape: (2726, 1)


...

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 3 Accuracy: 0.9259
Fold 3 Precision: 0.8474
Fold 3 Recall: 0.7909
Training Fold:  fold_4
X_train shape: (12014, 5), X_test shape: (2983, 5)
y_train shape: (12014, 1), y_test shape: (2983, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s


Fold 4 Accuracy: 0.9125
Fold 4 Precision: 0.8859
Fold 4 Recall: 0.7323
Metrics saved to results/SFold_PermafrostPrediction/RF-1_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


## Experiment Random Forest 2 (RF-2)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34
    - Covariates: Lat, Lon
    - Intuition: Basically all the midsummer channels
    - Order: B, G, R, rededge1, rededge2, rededge3, nir, swir1, swir2, lon, lat
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [10]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (14999, 72)
After removing NaN:  (14999, 72)


In [11]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

X shape:  (14999, 11)
Y shape:  (14999, 1)


In [12]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [13]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = True, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold/RF-2_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (12062, 11), X_test shape: (2937, 11)
y_train shape: (12062, 1), y_test shape: (2937, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.9091
Fold 0 Precision: 0.8581
Fold 0 Recall: 0.7418
2937
7548
Test predictions saved to test_predictions_fold_0.csv
Model saved to rf_weights_fold_0.pkl
Training Fold:  fold_1
X_train shape: (11777, 11), X_test shape: (3222, 11)
y_train shape: (11777, 1), y_test shape: (3222, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.5s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 1 Accuracy: 0.9119
Fold 1 Precision: 0.8066
Fold 1 Recall: 0.7308
3222
7409
Test predictions saved to test_predictions_fold_1.csv
Model saved to rf_weights_fold_1.pkl
Training Fold:  fold_2
X_train shape: (12273, 11), X_test shape: (2726, 11)
y_train shape: (12273, 1), y_test shape: (2726, 1)


...

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.5s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 3 Accuracy: 0.9208
Fold 3 Precision: 0.8492
Fold 3 Recall: 0.7591
3131
7790
Test predictions saved to test_predictions_fold_3.csv
Model saved to rf_weights_fold_3.pkl
Training Fold:  fold_4
X_train shape: (12016, 11), X_test shape: (2983, 11)
y_train shape: (12016, 1), y_test shape: (2983, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s


Fold 4 Accuracy: 0.9055
Fold 4 Precision: 0.8858
Fold 4 Recall: 0.6990
2983
7476
Test predictions saved to test_predictions_fold_4.csv
Model saved to rf_weights_fold_4.pkl
Metrics saved to results/SFold/RF-2_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.5s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


## Experiment Random Forest 3 (RF-3)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, (bands['s2_1_nir'] - bands['s2_1_red']) / (bands['s2_1_nir'] + bands['s2_1_red'])
    - Covariates: Lat, Lon
    - Order: B, G, R, NDVI
    - Intuition: Best performing channels + one natural derivative
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [22]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (14999, 79)
After removing NaN:  (14999, 79)


In [23]:
#Calculating the derivative channels
point_data_gdf['ndvi'] = (point_data_gdf['band_31'] - point_data_gdf['band_27']) / (point_data_gdf['band_31'] + point_data_gdf['band_27'])

In [24]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'ndvi']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

X shape:  (14999, 6)
Y shape:  (14999, 1)


In [25]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [26]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/RF-3_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (11992, 6), X_test shape: (3007, 6)
y_train shape: (11992, 1), y_test shape: (3007, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 0 Accuracy: 0.9395
Fold 0 Precision: 0.9099
Fold 0 Recall: 0.8015
Training Fold:  fold_1
X_train shape: (11983, 6), X_test shape: (3016, 6)
y_train shape: (11983, 1), y_test shape: (3016, 1)


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 1 Accuracy: 0.9344
Fold 1 Precision: 0.8983
Fold 1 Recall: 0.7934
Training Fold:  fold_2
X_train shape: (12003, 6), X_test shape: (2996, 6)
y_train shape: (12003, 1), y_test shape: (2996, 1)


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 2 Accuracy: 0.9269
Fold 2 Precision: 0.8841
Fold 2 Recall: 0.7923
Training Fold:  fold_3
X_train shape: (12029, 6), X_test shape: (2970, 6)
y_train shape: (12029, 1), y_test shape: (2970, 1)


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 3 Accuracy: 0.9303
Fold 3 Precision: 0.8838
Fold 3 Recall: 0.7988
Training Fold:  fold_4
X_train shape: (11989, 6), X_test shape: (3010, 6)
y_train shape: (11989, 1), y_test shape: (3010, 1)
Fold 4 Accuracy: 0.9302
Fold 4 Precision: 0.9065
Fold 4 Recall: 0.7791
Metrics saved to results/RF-3_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


## Experiment Random Forest 4 (RF-4)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27,
                (bands['s2_1_nir'] - bands['s2_1_swir2']) / (bands['s2_1_nir'] + bands['s2_1_swir2']),
                (bands['s2_1_green'] - bands['s2_1_red']) / (bands['s2_1_green'] + bands['s2_1_red']),
                (bands['s2_1_nir'] - bands['s2_1_swir1']) / (bands['s2_1_nir'] + bands['s2_1_swir1']),
                (bands['s2_1_green'] - bands['s2_1_swir1']) / (bands['s2_1_green'] + bands['s2_1_swir1']),
                (bands['s2_1_nir'] - bands['s2_1_red']) / (bands['s2_1_nir'] + bands['s2_1_red']),
                (bands['s2_1_green'] - bands['s2_1_nir']) / (bands['s2_1_green'] + bands['s2_1_nir'])
    - Covariates: Lat, Lon
    - Order: B, G, R, NBR, NGRDI, NDMI, NDSI, NDVI, NDWI, Lon, Lat
    - Intuition: Best performing channels + all derivatives
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [14]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (14999, 72)
After removing NaN:  (14999, 72)


In [15]:
#Calculating the derivative channels
point_data_gdf['nbr'] = (point_data_gdf['band_31'] - point_data_gdf['band_34']) / (point_data_gdf['band_31'] + point_data_gdf['band_34'])
point_data_gdf['ngrdi'] = (point_data_gdf['band_26'] - point_data_gdf['band_27']) / (point_data_gdf['band_26'] + point_data_gdf['band_27'])
point_data_gdf['ndmi'] = (point_data_gdf['band_31'] - point_data_gdf['band_33']) / (point_data_gdf['band_31'] + point_data_gdf['band_33'])
point_data_gdf['ndsi'] = (point_data_gdf['band_26'] - point_data_gdf['band_33']) / (point_data_gdf['band_26'] + point_data_gdf['band_33'])
point_data_gdf['ndvi'] = (point_data_gdf['band_31'] - point_data_gdf['band_27']) / (point_data_gdf['band_31'] + point_data_gdf['band_27'])
point_data_gdf['ndwi'] = (point_data_gdf['band_26'] - point_data_gdf['band_31']) / (point_data_gdf['band_26'] + point_data_gdf['band_31'])

In [16]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'nbr', 'ngrdi', 'ndmi', 'ndsi', 'ndvi', 'ndwi']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

X shape:  (14999, 11)
Y shape:  (14999, 1)


In [17]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-4_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (12062, 11), X_test shape: (2937, 11)
y_train shape: (12062, 1), y_test shape: (2937, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.9077
Fold 0 Precision: 0.8620
Fold 0 Recall: 0.7304
Training Fold:  fold_1
X_train shape: (11777, 11), X_test shape: (3222, 11)
y_train shape: (11777, 1), y_test shape: (3222, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 1 Accuracy: 0.9072
Fold 1 Precision: 0.8080
Fold 1 Recall: 0.6980
Training Fold:  fold_2
X_train shape: (12273, 11), X_test shape: (2726, 11)
y_train shape: (12273, 1), y_test shape: (2726, 1)


...

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 3 Accuracy: 0.9186
Fold 3 Precision: 0.8426
Fold 3 Recall: 0.7545
Training Fold:  fold_4
X_train shape: (12016, 11), X_test shape: (2983, 11)
y_train shape: (12016, 1), y_test shape: (2983, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s


Fold 4 Accuracy: 0.9001
Fold 4 Precision: 0.8704
Fold 4 Recall: 0.6893
Metrics saved to results/SFold_PermafrostPrediction/RF-4_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


## Experiment Random Forest 5 (RF-5)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: None
    - Covariates: Lat, Lon
    - Order: N/A
    - Intuition: Just Lat/Lon
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None

In [18]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (14999, 78)
After removing NaN:  (14999, 78)


In [19]:
# Loading in data 
channel_names = []
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

X shape:  (14999, 2)
Y shape:  (14999, 1)


In [20]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-5_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (12062, 2), X_test shape: (2937, 2)
y_train shape: (12062, 1), y_test shape: (2937, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 0 Accuracy: 0.9145
Fold 0 Precision: 0.8024
Fold 0 Recall: 0.8516
Training Fold:  fold_1
X_train shape: (11777, 2), X_test shape: (3222, 2)
y_train shape: (11777, 1), y_test shape: (3222, 1)


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 1 Accuracy: 0.9261
Fold 1 Precision: 0.7944
Fold 1 Recall: 0.8466
Training Fold:  fold_2
X_train shape: (12273, 2), X_test shape: (2726, 2)
y_train shape: (12273, 1), y_test shape: (2726, 1)


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 2 Accuracy: 0.9303
Fold 2 Precision: 0.8742
Fold 2 Recall: 0.8407
Training Fold:  fold_3
X_train shape: (11868, 2), X_test shape: (3131, 2)
y_train shape: (11868, 1), y_test shape: (3131, 1)


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s


Fold 3 Accuracy: 0.9157
Fold 3 Precision: 0.7765
Fold 3 Recall: 0.8424
Training Fold:  fold_4
X_train shape: (12016, 2), X_test shape: (2983, 2)
y_train shape: (12016, 1), y_test shape: (2983, 1)
Fold 4 Accuracy: 0.9293
Fold 4 Precision: 0.8795
Fold 4 Recall: 0.8197
Metrics saved to results/SFold_PermafrostPrediction/RF-5_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


## Experiment Random Forest 6 (RF-6)
### Settings:
- Spatial Scale: Buffer level - 100 m
- Inputs: 
    - Channels: R,G,B
    - Covariates: Lat, Lon (center)
    - Order: B, G, R, Lat, Lon
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: None


In [39]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (14999, 73)
After removing NaN:  (14999, 73)


In [40]:
# Project gdf to EPSG 3338 since we are going to do buffer operations
point_data_gdf = point_data_gdf.to_crs(epsg=3338)

In [41]:
# Loading in data
# For every row in the dataframe, get all the rows that are within a 100 meter buffer
# For all the rows in a 100 meter buffer get the average r, average g, and average b
point_data_gdf['100m_buffer'] = point_data_gdf.geometry.buffer(100)
buffers_gdf = point_data_gdf.set_geometry('100m_buffer')

sjoin_result = gpd.sjoin(point_data_gdf, buffers_gdf[['100m_buffer']], predicate="within")

# Group by buffer geometry and calculate the mean for each band
band_means = (
    sjoin_result.groupby(sjoin_result.index_right)[['band_25', 'band_26', 'band_27']]
    .mean()
    .reset_index()
)

# Rename columns for clarity
band_means = band_means.rename(
    columns={
        'band_25': 'ave_band_25',
        'band_26': 'ave_band_26',
        'band_27': 'ave_band_27'
    }
)

# Merge the calculated averages back into the buffers GeoDataFrame
buffers_points_gdf = buffers_gdf.merge(band_means, left_index=True, right_on='index_right', how='left')
print(buffers_points_gdf.columns)
# averages = (
#     buffered_gdf.groupby('index_left')
#     .agg(avg_r=('r', 'mean'), avg_g=('g', 'mean'), avg_b=('b', 'mean'))
# )
# point_data_gdf = point_data_gdf.join(averages)
# print(point_data_gdf.head(2))

Index(['id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'x_pixel', 'y_pixel', 'band_1', 'band_2',
       'band_3', 'band_4', 'band_5', 'band_6', 'band_7', 'band_8', 'band_9',
       'band_10', 'band_11', 'band_12', 'band_13', 'band_14', 'band_15',
       'band_16', 'band_17', 'band_18', 'band_19', 'band_20', 'band_21',
       'band_22', 'band_23', 'band_24', 'band_25', 'band_26', 'band_27',
       'band_28', 'band_29', 'band_30', 'band_31', 'band_32', 'band_33',
       'band_34', 'band_35', 'band_36', 'band_37', 'band_38', 'band_39',
       'band_40', 'band_41', 'band_42', 'band_43', 'band_44', 'band_45',
       'band_46', 'band_47', 'band_48', 'band_49', 'band_50', 'band_51',
       'band_52', 'band_53', 'band_54', 'band_55', 'band_56', 'band_57',
       'band_58', 'band_59', 'band_60', 'aksdb_pf1m_bin', 'aksdb_pf1m_bin_gt',
       'geometry', '100m_buffer', 'index_right', 'ave_band_25', 'ave_band_26',
       'ave_band_27'],
      dtype='object')


In [42]:
# Loading in data 
channel_names = ['ave_band_25', 'ave_band_26', 'ave_band_27']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = buffers_points_gdf[x_vars].to_numpy()
y = buffers_points_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = buffers_points_gdf['id'].to_list()

X shape:  (14999, 5)
Y shape:  (14999, 1)


In [44]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [45]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/RF-6_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (11992, 5), X_test shape: (3007, 5)
y_train shape: (11992, 1), y_test shape: (3007, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.9425
Fold 0 Precision: 0.9113
Fold 0 Recall: 0.8153
Training Fold:  fold_1
X_train shape: (11983, 5), X_test shape: (3016, 5)
y_train shape: (11983, 1), y_test shape: (3016, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 1 Accuracy: 0.9486
Fold 1 Precision: 0.9226
Fold 1 Recall: 0.8383
Training Fold:  fold_2
X_train shape: (12003, 5), X_test shape: (2996, 5)
y_train shape: (12003, 1), y_test shape: (2996, 1)


...

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 3 Accuracy: 0.9431
Fold 3 Precision: 0.8918
Fold 3 Recall: 0.8536
Training Fold:  fold_4
X_train shape: (11989, 5), X_test shape: (3010, 5)
y_train shape: (11989, 1), y_test shape: (3010, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s


Fold 4 Accuracy: 0.9465
Fold 4 Precision: 0.9365
Fold 4 Recall: 0.8250
Metrics saved to results/RF-6_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


## Experiment Random Forest 7 (RF-7)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lat, Lon
    - Covariates: Lat, Lon
    - Order: B, G, R, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Permafrost presence prediction
- Preprocessing: removal of NoData portions of sentinel data


In [9]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (14997, 72)
After removing NaN:  (14997, 72)


In [10]:
# Loading in covars
covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

Index(['field_1', 'id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'tax_order', 'x_pixel', 'y_pixel', 'grid_id',
       'aksdb_pf1m_bin', 'aspct_4_band_1', 'maxc_4_band_1', 'spi_band_1',
       'tpi_4_band_1', 'elevation_full_10m_3338_band_1', 'sl_4_band_1',
       'swi_10_band_1'],
      dtype='object')


In [11]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [13]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

X shape:  (14997, 12)
Y shape:  (14997, 1)


In [14]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [15]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-7_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (12060, 12), X_test shape: (2937, 12)
y_train shape: (12060, 1), y_test shape: (2937, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.9125
Fold 0 Precision: 0.8750
Fold 0 Recall: 0.7389
Training Fold:  fold_1
X_train shape: (11777, 12), X_test shape: (3220, 12)
y_train shape: (11777, 1), y_test shape: (3220, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.9s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 1 Accuracy: 0.9137
Fold 1 Precision: 0.8229
Fold 1 Recall: 0.7199
Training Fold:  fold_2
X_train shape: (12271, 12), X_test shape: (2726, 12)
y_train shape: (12271, 1), y_test shape: (2726, 1)


...

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    2.5s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 3 Accuracy: 0.9221
Fold 3 Precision: 0.8490
Fold 3 Recall: 0.7667
Training Fold:  fold_4
X_train shape: (12014, 12), X_test shape: (2983, 12)
y_train shape: (12014, 1), y_test shape: (2983, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s


Fold 4 Accuracy: 0.8984
Fold 4 Precision: 0.8814
Fold 4 Recall: 0.6699
Metrics saved to results/SFold_PermafrostPrediction/RF-7_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


## Experiment Random Forest 11 (RF-11)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: basically satellite channels, env covars like DEM, climate covars
        - B, G, R, rededge1, rededge2, rededge3, nir, swir1, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Presence/Absence of Permafrost, aksdb_pf1m_bin (image channel), binary classification
- Preprocessing: removal of NoData rows

In [5]:
print('Before removing NaN: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf.dropna(subset=['aksdb_pf1m_bin_gt']).reset_index(drop=True)
print('After removing NaN: ', point_data_gdf.shape)

Before removing NaN:  (37249, 72)
After removing NaN:  (14997, 72)


In [6]:
# Loading in covars
topo_covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
topo_covar_gdf = gpd.read_file(topo_covar_pt)
topo_covar_gdf = topo_covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]
topo_covar_gdf['id'] = topo_covar_gdf['id'].astype('int64')

climate_covar_pt = '../data/point_pixel_climate_v1.csv'
climate_covar_gdf = pd.read_csv(climate_covar_pt)
climate_covar_gdf = climate_covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

In [7]:
merged_df = point_data_gdf.merge(topo_covar_gdf, on='id', how='inner')
merged_df = merged_df.merge(climate_covar_gdf, on='id', how='inner')

In [8]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 
                    'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 
                     'swi_10_band_1', 'tpi_4_band_1', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['aksdb_pf1m_bin_gt']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (14997, 21)
Y shape:  (14997, 1)


In [9]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [10]:
# Run specific experiment settings
# indices, X, y, and hyperparameters should be specific to the variable of interest
# kfold_indices should remain the same among all experiments
(
    accuracy, precision, recall,
    precision_0, precision_1,
    recall_0, recall_1
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Precision_0": precision_0,
    "Recall_0": recall_0,
    "Precision_1": precision_1,
    "Recall_1": recall_1,
    
}
results_df = pd.DataFrame(results)
result_outpath = "results/SFold_PermafrostPrediction/RF-11_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (12060, 21), X_test shape: (2937, 21)
y_train shape: (12060, 1), y_test shape: (2937, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    2.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 0 Accuracy: 0.9196
Fold 0 Precision: 0.8894
Fold 0 Recall: 0.7575
Model saved to rf_weights_fold_0.pkl
Training Fold:  fold_1
X_train shape: (11777, 21), X_test shape: (3220, 21)
y_train shape: (11777, 1), y_test shape: (3220, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    2.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 1 Accuracy: 0.9236
Fold 1 Precision: 0.8466
Fold 1 Recall: 0.7512
Model saved to rf_weights_fold_1.pkl
Training Fold:  fold_2
X_train shape: (12271, 21), X_test shape: (2726, 21)
y_train shape: (12271, 1), y_test shape: (2726, 1)


...

Fold 2 Accuracy: 0.9193
Fold 2 Precision: 0.8742
Fold 2 Recall: 0.7891
Model saved to rf_weights_fold_2.pkl
Training Fold:  fold_3
X_train shape: (11866, 21), X_test shape: (3131, 21)
y_train shape: (11866, 1), y_test shape: (3131, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 3 Accuracy: 0.9310
Fold 3 Precision: 0.8663
Fold 3 Recall: 0.7955
Model saved to rf_weights_fold_3.pkl
Training Fold:  fold_4
X_train shape: (12014, 21), X_test shape: (2983, 21)
y_train shape: (12014, 1), y_test shape: (2983, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 4 Accuracy: 0.9199
Fold 4 Precision: 0.9213
Fold 4 Recall: 0.7309
Model saved to rf_weights_fold_4.pkl
Metrics saved to results/SFold_PermafrostPrediction/RF-11_results.csv
